In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Load tables

opp = spark.read.table("subscription_pipeline.silver.opportunity")
fx = spark.read.table("subscription_pipeline.silver.fx_rates")

# Clean + fix

opp_clean = opp.select(
"opportunity_id",
"customer_id",
"product_id",
"employee_id",
to_date("start_date", "dd-MM-yyyy").alias("start_date"),
to_date("end_date", "dd-MM-yyyy").alias("end_date"),
col("revenue_amount").cast("double"),
"close_status",
# to_timestamp("created_timestamp", "dd-MM-yyyy HH:mm").alias("created_timestamp")
)

In [0]:
display(opp_clean)

In [0]:
# FIXED FILTER

opp_clean = opp_clean.filter(
lower(col("close_status")) == 'won'
)

# Remove bad dates

opp_clean = opp_clean.filter(
col("start_date").isNotNull() &
col("end_date").isNotNull()
)

display(opp_clean)

In [0]:
# Expand monthly

fact = opp_clean.withColumn(
"date",
explode(sequence(col("start_date"), col("end_date"), expr("interval 1 month")))
)

# Join FX

fact = fact.join(
fx,
fact.date == fx.effective_date,
"left"
)

display(fact)

In [0]:
# Handle null FX

fact = fact.withColumn(
"fx_rate_to_gbp",
coalesce(col("fx_rate_to_gbp"), lit(1.0))
)

# Revenue GBP

fact = fact.withColumn(
"revenue_gbp",
col("revenue_amount") * col("fx_rate_to_gbp")
)

# Window

window_spec = Window.partitionBy("customer_id").orderBy("date")

fact = fact.withColumn("prev_revenue", lag("revenue_gbp").over(window_spec))

# Flags

fact = fact.withColumn("is_new_customer", when(col("prev_revenue").isNull(), 1).otherwise(0))
fact = fact.withColumn("is_churned", when(col("revenue_gbp") == 0, 1).otherwise(0))
fact = fact.withColumn("is_expansion", when(col("revenue_gbp") > col("prev_revenue"), 1).otherwise(0))
fact = fact.withColumn("is_contraction", when(col("revenue_gbp") < col("prev_revenue"), 1).otherwise(0))

# Final

fact_final = fact.select(
"customer_id",
"product_id",
"employee_id",
col("date").alias("revenue_date"),
"revenue_amount",
"revenue_gbp",
"is_new_customer",
"is_churned",
"is_expansion",
"is_contraction"
)

# display(fact_final)

fact_final.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("subscription_pipeline.gold.fact_subscription_revenue")

print("fixed fact table created")